# SurfaceTarget-CRC — phase 06 — patch 6 3

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 6 — PATCH 6.3
#
# DIRECT LAPTOP BACKUP AND GITHUB PREPARATION
#
# OBJECTIVES:
# 1. Verify essential research results
# 2. Prepare a GitHub-ready research repository
# 3. Generate README and data-access documentation
# 4. Preserve selected scientific outputs
# 5. Create a complete backup of the Colab directory
# 6. Split large backups into browser-friendly parts
# 7. Generate a Windows reconstruction script
# 8. Record checksums for archive verification
#
# IMPORTANT:
# - NO GOOGLE DRIVE
# - NO AUTOMATIC GITHUB UPLOAD
# - NO MODIFICATION OF ORIGINAL RESEARCH DATA
# - COMPLETE BACKUP INCLUDES RAW DATA
# - PUBLIC PACKAGE EXCLUDES RAW DATA
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import shutil
import hashlib
import json
import re


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

if not ROOT.is_dir():
    raise FileNotFoundError(
        "SurfaceTarget-CRC directory not found."
    )

STAMP = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

EXPORT = Path(
    f"/content/SurfaceTarget_CRC_Export_{STAMP}"
)

EXPORT.mkdir(
    parents=True,
    exist_ok=True
)

REPO = EXPORT / "SurfaceTarget-CRC"

REPO.mkdir(
    parents=True,
    exist_ok=True
)

# Browser-friendly download chunk size.
PART_SIZE_MIB = 190

PART_SIZE_BYTES = PART_SIZE_MIB * 1024**2


print("=" * 68)
print("SURFACETARGET-CRC")
print("PHASE 6 — PATCH 6.3")
print("DIRECT SSD BACKUP AND GITHUB PREPARATION")
print("=" * 68)


# ------------------------------------------------------------
# 2. HELPERS
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def check_file(relative):

    path = ROOT / relative

    if not path.is_file():
        raise FileNotFoundError(path)

    return path


def copy_file(relative, required=False):

    source = ROOT / relative

    if not source.is_file():

        if required:
            raise FileNotFoundError(source)

        return False

    if source.is_symlink():
        raise RuntimeError(
            f"Unexpected symlink in public export: {source}"
        )

    destination = REPO / relative

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(source, destination)

    return True


# ------------------------------------------------------------
# 3. VALIDATE RESEARCH
# ------------------------------------------------------------

print("\nVALIDATING COMPLETED RESEARCH")


ESSENTIAL = [
    "reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md",

    "results/tables/surfacetarget_phase5_research_panel_v1.csv",

    "results/tables/surfacetarget_phase5_full_universe_panel_audit.csv",

    "results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv",

    "results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png",

    "results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png",

    "results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png"
]


for relative in ESSENTIAL:
    check_file(relative)


panel = pd.read_csv(
    ROOT /
    "results/tables/surfacetarget_phase5_research_panel_v1.csv"
)

universe = pd.read_csv(
    ROOT /
    "results/tables/surfacetarget_phase5_full_universe_panel_audit.csv",
    usecols=["gene_id"]
)


if len(panel) != 15:
    raise RuntimeError("Expected 15 research-panel genes.")

if len(universe) != 948:
    raise RuntimeError("Expected 948 research candidates.")

if universe["gene_id"].duplicated().any():
    raise RuntimeError("Duplicate research gene identifiers.")


print("\nResearch panel:", len(panel))
print("Full candidate universe:", len(universe))
print("Research validation: PASSED")


# ------------------------------------------------------------
# 4. PREPARE GITHUB DIRECTORY
# ------------------------------------------------------------

print("\nPREPARING GITHUB REPOSITORY")


for folder in [
    "reports",
    "results/tables",
    "results/figures",
    "data/metadata",
    "src",
    "notebooks",
    "docs"
]:

    (REPO / folder).mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 5. COPY SCIENTIFIC REPORTS
# ------------------------------------------------------------

REPORT_FILES = [
    "reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md",

    "reports/SurfaceTarget_CRC_Phase5_Research_Panel.md",

    "reports/SurfaceTarget_CRC_Phase6_Evidence_and_Figure_Audit.md"
]


for relative in REPORT_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 6. COPY RESEARCH FIGURES
# ------------------------------------------------------------

FIGURE_FILES = [
    "results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png",

    "results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png",

    "results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png",

    "results/figures/depmap_26q3_crc_dependency_comparison.png"
]


for relative in FIGURE_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 7. COPY SELECTED RESEARCH TABLES
# ------------------------------------------------------------

TABLE_FILES = [
    "results/tables/surfacetarget_phase5_research_panel_v1.csv",

    "results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv",

    "results/tables/surfacetarget_phase5_panel_normal_tissue_review.csv",

    "results/tables/surfacetarget_phase5_leading_rank_sensitivity.csv",

    "results/tables/surfacetarget_phase5_scenario_rank_correlations.csv",

    "results/tables/surfacetarget_phase5_ranking_stability.csv",

    "results/tables/surfacetarget_phase5_topology_class_summary.csv",

    "results/tables/surfacetarget_phase5_data_coverage.csv",

    "results/tables/surfacetarget_phase5_followup_lane_summary.csv",

    "results/tables/surfacetarget_phase6_report_claim_audit.csv",

    "results/tables/surfacetarget_phase6_report_panel_summary.csv",

    "results/tables/surfacetarget_phase6_figure_manifest.csv"
]


for relative in TABLE_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 8. COPY SELECTED PROVENANCE
# ------------------------------------------------------------

METADATA_FILES = [
    "data/metadata/surfacetarget_phase5_research_panel_provenance.json",

    "data/metadata/surfacetarget_phase5_prioritization_provenance.json",

    "data/metadata/surfacetarget_phase5_robustness_provenance.json",

    "data/metadata/surfacetarget_phase6_figure_provenance.json",

    "data/metadata/surfacetarget_phase6_report_draft_provenance.json"
]


for relative in METADATA_FILES:
    copy_file(relative)


# ------------------------------------------------------------
# 9. COPY AVAILABLE SOURCE SCRIPTS
# ------------------------------------------------------------

print("\nCOPYING SOURCE CODE")


SAFE_EXTENSIONS = {
    ".py",
    ".r",
    ".sh",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".toml"
}


copied_scripts = 0

source_folder = ROOT / "src"


if source_folder.is_dir():

    for source in source_folder.rglob("*"):

        if not source.is_file():
            continue

        if source.is_symlink():
            continue

        if source.suffix.lower() not in SAFE_EXTENSIONS:
            continue

        if source.stat().st_size > 5 * 1024**2:
            continue

        relative = source.relative_to(ROOT)

        destination = REPO / relative

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(source, destination)

        copied_scripts += 1


print("Source scripts included:", copied_scripts)


if (ROOT / "requirements.txt").is_file():

    copy_file("requirements.txt")

else:

    (REPO / "requirements-reporting.txt").write_text(
        "pandas\n"
        "numpy\n"
        "matplotlib\n",
        encoding="utf-8"
    )


# ------------------------------------------------------------
# 10. CREATE GITHUB README
# ------------------------------------------------------------

print("\nCREATING README")


README = """# SurfaceTarget-CRC

## Multi-Omics Prioritization of Colorectal Cancer
## Cell-Surface and Microenvironment Target Hypotheses

SurfaceTarget-CRC is an exploratory computational biology
research project integrating transcriptomics, proteomics,
protein localization, normal-tissue annotations, and
CRISPR functional-genomics evidence.

The project develops evidence-supported research hypotheses
for experimental investigation. It does not claim
clinically validated therapeutic targets.

## Research data sources

- TCGA/GDC colorectal transcriptomics
- CPTAC paired MSS colon cancer proteomics
- Human Protein Atlas (HPA)
- UniProt reviewed protein annotations
- DepMap Public 26Q3 CRISPR gene-effect data
- HGNC gene identifier mappings

## Main findings

| Research component | Result |
|---|---:|
| Original research candidates | 948 |
| Qualifying CPTAC-enriched genes | 20 |
| Stronger surface-related topology annotations | 112 |
| Verified DepMap mappings | 943 |
| Putative cell-surface review candidates | 111 |
| Detailed research panel | 15 |

CA9, ADAM12, and NPC1L1 appeared in the
top 20 under all three research-prioritization scenarios.

However, the protein-first and normal-review-first
rankings were highly correlated (Spearman rho = 0.986).

Ranking consistency is not independent
experimental validation.

## Research figures

### Figure 1: Cross-omics evidence coverage

![Evidence coverage](results/figures/SurfaceTarget_CRC_Fig1_Evidence_Coverage.png)

### Figure 2: Protein topology

![Protein topology](results/figures/SurfaceTarget_CRC_Fig2_Topology_Distribution.png)

### Figure 3: Ranking sensitivity

![Ranking sensitivity](results/figures/SurfaceTarget_CRC_Fig3_Ranking_Sensitivity.png)

## Fifteen-gene research panel

**Cross-scenario surface hypotheses**

CA9, ADAM12, NPC1L1

**Additional surface-antigen hypotheses**

CDH3, CLDN1, DPEP1, SLC7A5, TDGF1, LGR5

**Stromal-targeting hypothesis**

FAP

**Membrane-accessibility investigations**

SLC5A6, CEMIP

**Extracellular microenvironment hypotheses**

CTHRC1, MXRA5, SFRP4

The groups describe different biological mechanisms.
They are not a single therapeutic ranking.

## Research documentation

[Formal scientific report](reports/SurfaceTarget_CRC_Formal_Research_Report_Draft_v1.md)

[Experimental validation roadmap](results/tables/surfacetarget_phase5_experimental_validation_roadmap.csv)

[Data access documentation](docs/DATA_ACCESS.md)

## Repository structure

- reports/ : Research interpretation and scientific report
- results/figures/ : Research visualizations
- results/tables/ : Selected derived analysis outputs
- data/metadata/ : Selected provenance files
- src/ : Available analysis scripts
- notebooks/ : Jupyter notebooks after manual review
- docs/ : Data access and reproducibility information

## Reproducibility

This repository contains a documented computational
research analysis and selected derived outputs.

The complete executed Colab notebook should be
added to notebooks/ after reviewing its contents.

Raw TCGA, CPTAC, and DepMap data are not
redistributed in this repository.

See docs/DATA_ACCESS.md for source information.

The report remains a technical draft. Full
independent reproducibility and publication readiness
require verification of earlier cohort methods,
dataset accessions, software versions, and citations.

## Limitations

- TCGA discovery cases are not uniformly MSS-confirmed.
- DepMap colorectal cell lines are not all MSS-confirmed.
- Bulk expression does not establish malignant-cell origin.
- Protein topology does not prove antibody accessibility.
- HPA normal-tissue staining does not establish safety.
- CRISPR dependency does not directly establish ADC efficacy.
- No new experimental validation was performed.

## Status

Exploratory computational research.

The findings represent target hypotheses,
not clinically validated therapeutic recommendations.
"""


(REPO / "README.md").write_text(
    README,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 11. CREATE DATA ACCESS DOCUMENTATION
# ------------------------------------------------------------

DATA_ACCESS = """# Data Access and Reproducibility

The complete original molecular datasets are not
redistributed in this GitHub repository.

## TCGA/GDC

TCGA-COAD and TCGA-READ colorectal
transcriptomic datasets.

## CPTAC

Proteomic Data Commons colon cancer data.

Project accession used:
PDC000116

## DepMap

Public 26Q3:

- Model.csv
- CRISPRGeneEffect.csv

The working CRISPR file was reconstructed from
upload fragments. Its local checksum was validated,
but the original release checksum was not
independently verified.

## Human Protein Atlas

Normal-tissue IHC and subcellular annotations.

## UniProt

Reviewed human protein records and topology features.

## HGNC

Gene symbol and identifier crosswalks.

## Reproducibility considerations

Release versions, original identifiers,
data-access permissions, and source-specific
licenses should be verified before reproduction.

The complete private backup contains the
original research directory, including raw data
and intermediate analyses.

The Colab notebook must be downloaded separately
if it is not already stored inside the project folder.

## Redistribution

Review the reuse and redistribution terms of each
source before publicly distributing derived data.

No third-party datasets are relicensed here.
"""


(
    REPO / "docs/DATA_ACCESS.md"
).write_text(
    DATA_ACCESS,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 12. CREATE GITIGNORE
# ------------------------------------------------------------

GITIGNORE = """# Raw and large datasets
data/raw/
data/processed/
checkpoints/
*.zip
*.tar
*.tar.gz
*.7z
*.part
*.part[0-9]*

# Credentials
.env
.env.*
*.pem
*.key
*credentials*
*secret*
*token*

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/
.pytest_cache/

# Local/runtime files
*.log
.DS_Store
Thumbs.db
"""


(REPO / ".gitignore").write_text(
    GITIGNORE,
    encoding="utf-8"
)


(
    REPO / "notebooks/README.md"
).write_text(

    "# Analysis Notebook\n\n"
    "Add the executed SurfaceTarget-CRC Colab notebook "
    "here after manually checking it for credentials, "
    "personal information and excessive embedded outputs.\n",

    encoding="utf-8"
)


# ------------------------------------------------------------
# 13. SECURITY AND SIZE AUDIT
# ------------------------------------------------------------

print("\nAUDITING GITHUB EXPORT")


SECRET_PATTERNS = [
    re.compile(r"ghp_[A-Za-z0-9]{20,}"),
    re.compile(r"github_pat_[A-Za-z0-9_]{20,}"),
    re.compile(r"sk-[A-Za-z0-9_-]{20,}"),
    re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----")
]


TEXT_EXTENSIONS = {
    ".py",
    ".md",
    ".txt",
    ".csv",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".r",
    ".sh"
}


repo_files = [
    path
    for path in REPO.rglob("*")
    if path.is_file()
]


for path in repo_files:

    if path.stat().st_size > 25 * 1024**2:
        raise RuntimeError(
            f"Large public file requires review: {path}"
        )

    if path.suffix.lower() in TEXT_EXTENSIONS:

        contents = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        for pattern in SECRET_PATTERNS:

            if pattern.search(contents):
                raise RuntimeError(
                    f"Potential secret found in: {path}"
                )


print("\nGitHub export files:", len(repo_files))
print("Large-file screening: PASSED")
print("Common secret-pattern screening: PASSED")
print("Manual privacy and licensing review still required.")


# ------------------------------------------------------------
# 14. CREATE GITHUB ZIP
# ------------------------------------------------------------

print("\nCREATING GITHUB ZIP")


GITHUB_ZIP = (
    EXPORT /
    f"SurfaceTarget_CRC_GitHub_{STAMP}.zip"
)


with ZipFile(
    GITHUB_ZIP,
    "w",
    ZIP_DEFLATED,
    compresslevel=3,
    allowZip64=True
) as archive:

    for path in REPO.rglob("*"):

        if path.is_file():

            archive.write(
                path,
                arcname=(
                    "SurfaceTarget-CRC/"
                    + path.relative_to(REPO).as_posix()
                )
            )


print("\nGitHub archive created.")

print(
    "Size:",
    f"{GITHUB_ZIP.stat().st_size / 1024**2:.2f} MiB"
)


# ------------------------------------------------------------
# 15. INVENTORY COMPLETE COLAB DIRECTORY
# ------------------------------------------------------------

print("\nINVENTORYING COMPLETE PROJECT")


all_files = sorted(
    [
        path
        for path in ROOT.rglob("*")
        if path.is_file() and not path.is_symlink()
    ],
    key=lambda p: str(p)
)


symlinks = [
    path
    for path in ROOT.rglob("*")
    if path.is_symlink()
]


total_bytes = sum(
    path.stat().st_size
    for path in all_files
)


print("\nFiles to back up:", len(all_files))

print(
    "Uncompressed project size:",
    f"{total_bytes / 1024**3:.3f} GiB"
)

print(
    "Symlinks not archived:",
    len(symlinks)
)

if symlinks:
    print(
        "WARNING: Symlinks require separate review "
        "if their targets contain additional project data."
    )


# ------------------------------------------------------------
# 16. CREATE COMPLETE PRIVATE BACKUP
# ------------------------------------------------------------

print("\nCREATING COMPLETE PRIVATE BACKUP")


FULL_ZIP = (
    EXPORT /
    f"SurfaceTarget_CRC_COMPLETE_{STAMP}.zip"
)


with ZipFile(
    FULL_ZIP,
    "w",
    ZIP_DEFLATED,
    compresslevel=1,
    allowZip64=True
) as archive:

    for index, path in enumerate(
        all_files,
        start=1
    ):

        archive.write(
            path,
            arcname=(
                ROOT.name
                + "/"
                + path.relative_to(ROOT).as_posix()
            )
        )

        if index % 100 == 0:

            print(
                f"Archived {index}/{len(all_files)} files"
            )


print("\nComplete project archive created.")

print(
    "Compressed size:",
    f"{FULL_ZIP.stat().st_size / 1024**3:.3f} GiB"
)


# ------------------------------------------------------------
# 17. VERIFY ARCHIVES
# ------------------------------------------------------------

print("\nVERIFYING ARCHIVES")


for archive_path in [
    GITHUB_ZIP,
    FULL_ZIP
]:

    with ZipFile(
        archive_path,
        "r"
    ) as archive:

        bad_file = archive.testzip()

        if bad_file is not None:
            raise RuntimeError(
                f"Corrupt archive member: {bad_file}"
            )

    print(
        "Verified:",
        archive_path.name
    )


# ------------------------------------------------------------
# 18. SPLIT LARGE COMPLETE BACKUP
# ------------------------------------------------------------

print("\nPREPARING DIRECT DOWNLOAD PARTS")


full_hash = sha256_file(FULL_ZIP)

download_parts = []


if FULL_ZIP.stat().st_size <= PART_SIZE_BYTES:

    download_parts = [FULL_ZIP]

    print("\nArchive fits in one browser download.")

else:

    with open(FULL_ZIP, "rb") as source:

        part_number = 1

        while True:

            first_chunk = source.read(
                min(PART_SIZE_BYTES, 8 * 1024**2)
            )

            if not first_chunk:
                break

            part_path = Path(
                str(FULL_ZIP)
                + f".part{part_number:03d}"
            )

            written = 0

            with open(part_path, "wb") as output:

                output.write(first_chunk)

                written += len(first_chunk)

                while written < PART_SIZE_BYTES:

                    chunk = source.read(
                        min(
                            8 * 1024**2,
                            PART_SIZE_BYTES - written
                        )
                    )

                    if not chunk:
                        break

                    output.write(chunk)

                    written += len(chunk)

            download_parts.append(part_path)

            print(
                f"Part {part_number}: "
                f"{written / 1024**2:.2f} MiB"
            )

            part_number += 1


# ------------------------------------------------------------
# 19. GENERATE WINDOWS REASSEMBLY SCRIPT
# ------------------------------------------------------------

print("\nPREPARING WINDOWS REASSEMBLY INSTRUCTIONS")


REASSEMBLE_BAT = (
    EXPORT /
    f"Reassemble_SurfaceTarget_CRC_{STAMP}.bat"
)


if len(download_parts) > 1:

    joined_names = "+".join(
        f'"{part.name}"'
        for part in download_parts
    )

    script = (
        "@echo off\n"
        "setlocal\n"
        'cd /d "%~dp0"\n'
        "echo Reassembling SurfaceTarget-CRC backup...\n"
        f'copy /b {joined_names} "{FULL_ZIP.name}"\n'
        "if errorlevel 1 (\n"
        "  echo Reconstruction failed.\n"
        "  pause\n"
        "  exit /b 1\n"
        ")\n"
        "echo.\n"
        "echo Expected SHA256:\n"
        f"echo {full_hash}\n"
        "echo.\n"
        "echo Actual SHA256:\n"
        f'certutil -hashfile "{FULL_ZIP.name}" SHA256\n'
        "echo.\n"
        "echo Confirm the two hashes match.\n"
        "pause\n"
    )

else:

    script = (
        "@echo off\n"
        "echo Your backup is already a complete ZIP.\n"
        "pause\n"
    )


REASSEMBLE_BAT.write_text(
    script,
    encoding="utf-8"
)


# ------------------------------------------------------------
# 20. CREATE DOWNLOAD MANIFEST
# ------------------------------------------------------------

print("\nCREATING DOWNLOAD MANIFEST")


MANIFEST_JSON = EXPORT / "download_manifest.json"


manifest = {
    "project": "SurfaceTarget-CRC",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "complete_archive_name": FULL_ZIP.name,
    "complete_archive_sha256": full_hash,
    "complete_archive_size_bytes": FULL_ZIP.stat().st_size,
    "part_size_mib": PART_SIZE_MIB,
    "download_parts": [
        {
            "name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path)
        }
        for path in download_parts
    ],
    "github_archive": {
        "name": GITHUB_ZIP.name,
        "size_bytes": GITHUB_ZIP.stat().st_size,
        "sha256": sha256_file(GITHUB_ZIP)
    },
    "original_research_directory_modified": False,
    "google_drive_used": False
}


MANIFEST_JSON.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)


# Verify split data reproduces the archive hash.

if len(download_parts) > 1:

    digest = hashlib.sha256()

    for part in download_parts:

        with open(part, "rb") as handle:

            for chunk in iter(
                lambda: handle.read(8 * 1024**2),
                b""
            ):
                digest.update(chunk)

    if digest.hexdigest() != full_hash:

        raise RuntimeError(
            "Backup parts failed reconstruction checksum."
        )

    # The original archive is no longer needed
    # in the temporary export area because the
    # validated parts contain identical bytes.
    # This does NOT delete any original project file.

    FULL_ZIP.unlink()


# ------------------------------------------------------------
# 21. VERIFY DOWNLOAD PACKAGE
# ------------------------------------------------------------

print("\nVERIFYING DOWNLOAD PACKAGE")


download_files = [
    GITHUB_ZIP,
    MANIFEST_JSON,
    REASSEMBLE_BAT,
    *download_parts
]


for path in download_files:

    if not path.is_file():

        raise RuntimeError(
            f"Missing download file: {path}"
        )


print("\nDownload package: VERIFIED")


# ------------------------------------------------------------
# 22. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("PATCH 6.3 COMPLETE")
print("=" * 68)


print("\nCOMPLETE PROJECT BACKUP")

print("Original research files:", len(all_files))

print(
    "Total original size:",
    f"{total_bytes / 1024**3:.3f} GiB"
)

print(
    "Complete backup parts:",
    len(download_parts)
)


print("\nGITHUB REPOSITORY")

print(
    "GitHub ZIP:",
    GITHUB_ZIP.name
)

print(
    "GitHub archive size:",
    f"{GITHUB_ZIP.stat().st_size / 1024**2:.2f} MiB"
)


print("\nFILES READY FOR DOWNLOAD:")

for index, path in enumerate(
    download_files,
    start=1
):

    print(
        f"{index}. {path.name} "
        f"({path.stat().st_size / 1024**2:.2f} MiB)"
    )


print("\nEXPORT DIRECTORY:")
print(EXPORT)


print("\nIMPORTANT:")
print("No Google Drive used.")
print("Original project directory unchanged.")
print("No GitHub upload performed.")
print("Save your Colab notebook separately.")
print("Download all parts before disconnecting Colab.")

print("=" * 68)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# GITHUB PUBLICATION — STEP 1
#
# PREPARE THE COMPLETE ANALYSIS NOTEBOOK
# ============================================================

from pathlib import Path
from google.colab import _message

import json
import re

REPO = Path(
    "/content/SurfaceTarget_CRC_Export_20261121_092801/"
    "SurfaceTarget-CRC"
)

if not REPO.is_dir():
    raise FileNotFoundError(
        "GitHub project folder not found. "
        "Check that Patch 6.3's export still exists."
    )

NOTEBOOKS = REPO / "notebooks"
NOTEBOOKS.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = (
    NOTEBOOKS /
    "SurfaceTarget_CRC_Complete_Analysis.ipynb"
)

print("=" * 65)
print("SURFACETARGET-CRC — NOTEBOOK EXPORT")
print("=" * 65)

print("\nRetrieving current Colab notebook...")

response = _message.blocking_request(
    "get_ipynb",
    timeout_sec=60
)

if not response:
    raise RuntimeError(
        "Could not retrieve the current notebook."
    )

notebook = response.get("ipynb", response)

if isinstance(notebook, str):
    notebook = json.loads(notebook)

if not isinstance(notebook, dict):
    raise RuntimeError(
        "Unexpected notebook response format."
    )

if "cells" not in notebook:
    raise RuntimeError(
        "Notebook response does not contain cells."
    )

# Remove execution outputs and runtime metadata.
# Preserve all Python and Markdown source cells.

clean_cells = []

for cell in notebook["cells"]:

    cell_type = cell.get("cell_type")

    if cell_type not in [
        "code",
        "markdown"
    ]:
        continue

    cleaned = {
        "cell_type": cell_type,
        "metadata": {},
        "source": cell.get("source", [])
    }

    if cell_type == "code":
        cleaned["execution_count"] = None
        cleaned["outputs"] = []

    clean_cells.append(cleaned)

clean_notebook = {
    "cells": clean_cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 4
}

# Check source cells for recognizable credential formats.

source_text = "\n".join(
    "".join(cell["source"])
    if isinstance(cell["source"], list)
    else str(cell["source"])
    for cell in clean_cells
)

secret_patterns = [
    r"ghp_[A-Za-z0-9]{30,}",
    r"github_pat_[A-Za-z0-9_]{30,}",
    r"sk-proj-[A-Za-z0-9_-]{30,}",
    r"-----BEGIN [A-Z ]*PRIVATE KEY-----"
]

for pattern in secret_patterns:
    if re.search(pattern, source_text):
        raise RuntimeError(
            "Potential credential found in notebook source. "
            "Review before uploading."
        )

NOTEBOOK_PATH.write_text(
    json.dumps(
        clean_notebook,
        indent=1,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\nNotebook saved successfully!")

print("Total cells:", len(clean_cells))

print(
    "Python cells:",
    sum(
        cell["cell_type"] == "code"
        for cell in clean_cells
    )
)

print(
    "Markdown cells:",
    sum(
        cell["cell_type"] == "markdown"
        for cell in clean_cells
    )
)

print(
    "Notebook size:",
    f"{NOTEBOOK_PATH.stat().st_size / 1024**2:.2f} MiB"
)

print("\nLocation:")
print(NOTEBOOK_PATH)

print("\nSaved execution outputs: REMOVED")
print("Source code: PRESERVED")
print("Original notebook: UNCHANGED")

print("\nNOTEBOOK EXPORT COMPLETE")